In [22]:
from pathlib import Path
import sys

root = Path.cwd()
if not (root / "src").is_dir():
    root = root.parent
sys.path.insert(0, str(root))

In [23]:
import numpy as np
from qiskit.quantum_info import Statevector
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score,recall_score,f1_score, classification_report
from src.data_load import load_data
from src.preprocess import preprocess

In [24]:
features=4
qubits=int(np.log2(features))

X_train, X_test, y_train, y_test= load_data()
X_train_scale,X_test_scale, X_train_pca, X_test_pca, pca=preprocess(X_train,X_test,qubits)

In [25]:
def normalize(X):
    return X/np.linalg.norm(X,axis=1, keepdims=True)

# X_train_amp= normalize(X_train_pca)
# X_test_amp=normalize(X_test_pca)

In [26]:
from sklearn.preprocessing import MinMaxScaler

mm = MinMaxScaler(feature_range=(0.1, 1))
X_train_amp = normalize(mm.fit_transform(X_train_pca))
X_test_amp = normalize(np.clip(mm.transform(X_test_pca), 0.1, 1))

In [27]:
def to_states(X):
    return np.array([Statevector(row).data for row in X])

S_train=to_states(X_train_amp)
S_test=to_states(X_test_amp)

In [28]:
K_train=np.abs(S_train.conj() @ S_train.T)**2
K_test=np.abs(S_test.conj() @ S_train.T)**2
print(K_train.shape, K_test.shape)

off_diagonal= K_train[~np.eye(len(K_train),dtype=bool)]
print("kernel off diagonal mean: ",off_diagonal.mean(), "std: ",off_diagonal.std())

(455, 455) (114, 455)
kernel off diagonal mean:  0.9018548711588474 std:  0.12675514054414605


In [29]:
qsvm=SVC(kernel="precomputed",C=1.0)
qsvm.fit(K_train,y_train)
y_pred_q=qsvm.predict(K_test)

print("QSVM amplitude accuracy: ",accuracy_score(y_test,y_pred_q))
print("Malignant recall:", recall_score(y_test, y_pred_q, pos_label=0))
print("F1 (macro):", f1_score(y_test, y_pred_q, average="macro"))
print(classification_report(y_test, y_pred_q, target_names=["malignant", "benign"]))

QSVM amplitude accuracy:  0.956140350877193
Malignant recall: 0.9523809523809523
F1 (macro): 0.9531057178116001
              precision    recall  f1-score   support

   malignant       0.93      0.95      0.94        42
      benign       0.97      0.96      0.97        72

    accuracy                           0.96       114
   macro avg       0.95      0.96      0.95       114
weighted avg       0.96      0.96      0.96       114



In [30]:
svm= SVC(kernel="rbf",C=1.0, gamma="scale").fit(X_train_amp, y_train)
y_pred_classical=svm.predict(X_test_amp)
print("Classical SVM (same data) accuracy:", accuracy_score(y_test, y_pred_classical))
print("Classical malignant recall:", recall_score(y_test, y_pred_classical, pos_label=0))
print("Classical F1 (macro):", f1_score(y_test, y_pred_classical, average="macro"))

Classical SVM (same data) accuracy: 0.9210526315789473
Classical malignant recall: 0.9523809523809523
Classical F1 (macro): 0.9170641015277665
